In [3]:
import json
import pandas as pd
from pathlib import Path

base = Path("data")
print(base.exists())
print(Path.cwd())

True
D:\Users\ghiattka\Documents\test_technique\Athlete_monitoring


In [6]:
with open(base / "p01" / "fitbit" / "steps.json", encoding="utf-8") as fh:
    d = json.load(fh)

df = pd.DataFrame(d)
df.head()

df["date"] = pd.to_datetime(df["dateTime"]).dt.normalize()
df["value"] = pd.to_numeric(df["value"])

steps_jour = df.groupby("date")["value"].sum()
print(steps_jour.head())
print(len(steps_jour), "jours")

date
2019-11-01    17873
2019-11-02    13118
2019-11-03    14312
2019-11-04    10970
2019-11-05    16186
Name: value, dtype: int64
152 jours


In [8]:
def charger_flux_minute(chemin, nom, agg="sum"):
    """Charge un JSON minute et agrège au jour."""
    with open(chemin, encoding="utf-8") as fh:
        d = json.load(fh)
    df = pd.DataFrame(d)
    df["date"] = pd.to_datetime(df["dateTime"]).dt.normalize()
    df["value"] = pd.to_numeric(df["value"])
    return df.groupby("date")["value"].agg(agg).rename(nom)

In [9]:
p = base / "p01" / "fitbit"
steps = charger_flux_minute(p / "steps.json", "steps")
cal = charger_flux_minute(p / "calories.json", "calories")
dist = charger_flux_minute(p / "distance.json", "distance_cm")

print(pd.concat([steps, cal, dist], axis=1).head())

            steps  calories  distance_cm
date                                    
2019-11-01  17873   4009.10      1442400
2019-11-02  13118   3533.56      1058480
2019-11-03  14312   3748.73      1146085
2019-11-04  10970   3353.38       885970
2019-11-05  16186   3794.63      1371470


In [10]:
with open(p / "heart_rate.json", encoding="utf-8") as fh:
    d = json.load(fh)

hr = pd.DataFrame(d)
hr["date"] = pd.to_datetime(hr["dateTime"]).dt.normalize()
hr["bpm"] = hr["value"].apply(lambda v: v["bpm"])

hr_jour = hr.groupby("date")["bpm"].agg(["mean", "min", "max", "std"])
hr_jour.head()

,mean,min,max,std
date,,,,
2019-11-01,66.140075,46,140,15.080539
2019-11-02,62.602226,44,122,15.280065
2019-11-03,64.551841,45,132,14.532847
2019-11-04,63.552109,44,126,13.494872
2019-11-05,66.672530,44,163,23.635823


In [11]:
with open(p / "resting_heart_rate.json", encoding="utf-8") as fh:
    d = json.load(fh)

rhr = pd.DataFrame(d)
rhr["date"] = pd.to_datetime(rhr["dateTime"]).dt.normalize()
rhr["rhr"] = rhr["value"].apply(lambda v: v["value"])
rhr_jour = rhr.set_index("date")["rhr"]

In [13]:
with open(p / "sleep.json", encoding="utf-8") as fh:
    d = json.load(fh)

sl = pd.DataFrame(d)
sl["date"] = pd.to_datetime(sl["dateOfSleep"])
sl["deep_min"] = sl["levels"].apply(lambda v: v["summary"].get("deep", {}).get("minutes"))
sl["rem_min"] = sl["levels"].apply(lambda v: v["summary"].get("rem", {}).get("minutes"))

with open(p / "sleep.json", encoding="utf-8") as fh:
    d = json.load(fh)

sl = pd.DataFrame(d)
sl["date"] = pd.to_datetime(sl["dateOfSleep"])
sl["deep_min"] = sl["levels"].apply(lambda v: v["summary"].get("deep", {}).get("minutes"))
sl["rem_min"] = sl["levels"].apply(lambda v: v["summary"].get("rem", {}).get("minutes"))

In [14]:
with open(p / "exercise.json", encoding="utf-8") as fh:
    d = json.load(fh)

ex = pd.DataFrame(d)
ex["date"] = pd.to_datetime(ex["startTime"]).dt.normalize()
print(ex[["date", "activityName", "duration", "calories", "steps", "averageHeartRate"]].head())

        date activityName  duration  calories  steps  averageHeartRate
0 2019-11-01         Walk   1331000       192   1878                94
1 2019-11-01         Walk   2202000       302   2786                94
2 2019-11-02         Walk   2458000       354   3035                98
3 2019-11-04         Walk   1024000       145   1284                97
4 2019-11-05         Walk    973000       121   1065                93


In [15]:
pm = base / "p01" / "pmsys"
wellness = pd.read_csv(pm / "wellness.csv")
srpe = pd.read_csv(pm / "srpe.csv")
injury = pd.read_csv(pm / "injury.csv")

for nom, df_ in [("wellness", wellness), ("srpe", srpe), ("injury", injury)]:
    print(f"\n{nom} : {df_.shape}")
    print(df_.head(3))


wellness : (138, 9)
       effective_time_frame  fatigue  mood  readiness  sleep_duration_h  \
0  2019-11-01T08:31:40.751Z        2     3          5                 6   
1  2019-11-02T10:00:01.229Z        2     3          6                 6   
2  2019-11-03T14:28:03.263Z        3     3          8                 6   

   sleep_quality  soreness soreness_area  stress  
0              3         2    [12921003]       3  
1              3         2    [12921003]       3  
2              3         3            []       3  

srpe : (34, 4)
              end_date_time             activity_names  perceived_exertion  \
0  2019-11-05T22:51:54.710Z  ['individual', 'running']                   7   
1  2019-11-11T21:15:15.092Z  ['individual', 'running']                   6   
2  2019-11-14T21:00:53.000Z         ['team', 'soccer']                   7   

   duration_min  
0            30  
1            30  
2            60  

injury : (24, 2)
       effective_time_frame injuries
0  2019-11-07T06:3

In [16]:
injury["a_blessure"] = injury["injuries"] != "{}"
print(injury["a_blessure"].sum(), "déclarations avec blessure sur", len(injury))
print(injury[injury["a_blessure"]])

1 déclarations avec blessure sur 24
        effective_time_frame                 injuries  a_blessure
11  2020-01-07T22:33:38.989Z  {'right_hand': 'minor'}        True


In [17]:
for pid in ["p01", "p03", "p05"]:
    inj = pd.read_csv(base / pid / "pmsys" / "injury.csv")
    inj["a_blessure"] = inj["injuries"] != "{}"
    print(f"{pid} : {inj['a_blessure'].sum()} blessure(s) sur {len(inj)} déclarations")
    if inj["a_blessure"].sum():
        print(inj[inj["a_blessure"]][["effective_time_frame", "injuries"]].to_string(index=False))

p01 : 1 blessure(s) sur 24 déclarations
    effective_time_frame                injuries
2020-01-07T22:33:38.989Z {'right_hand': 'minor'}
p03 : 0 blessure(s) sur 11 déclarations
p05 : 10 blessure(s) sur 10 déclarations
    effective_time_frame               injuries
2019-11-01T06:30:35.565Z {'left_foot': 'minor'}
2019-11-07T07:29:02.805Z {'head_neck': 'minor'}
2020-01-30T06:38:02.691Z {'left_foot': 'minor'}
2020-01-31T05:40:48.569Z {'left_foot': 'minor'}
2020-02-08T04:59:19.621Z {'left_foot': 'minor'}
2020-02-14T05:32:10.455Z {'left_foot': 'minor'}
2020-02-29T06:26:35.754Z {'left_foot': 'minor'}
2020-03-07T08:01:19.947Z {'left_foot': 'minor'}
2020-03-15T06:01:04.240Z {'left_foot': 'minor'}
2020-03-17T19:47:51.329Z {'left_foot': 'minor'}
